# FL Project: Optimization Algorithm Comparison
## 1. Data Loading

In [ ]:
import sys
sys.path.append("..")  # so Python can find the src/ folder

from src.get_data import fetch_data

df = fetch_data(cache_path="../data/daily_records.csv")
df.head()

In [ ]:
from src.graph import get_station_index, build_knn_graph, graph_summary

stations = get_station_index(df)
A, edge_index = build_knn_graph(stations, k=3, weight="distance")
graph_summary(stations, A, edge_index)

In [ ]:
from src.graph import get_station_index
from src.models import build_local_datasets, init_weights, mse, local_gradient

stations = get_station_index(df)
data = build_local_datasets(df, stations)
weights = init_weights(data)

In [ ]:
from src.graph import get_station_index, build_knn_graph
from src.models import build_local_datasets, init_weights, mse
from src.fl_algorithm import run_a2, run_a4

stations = get_station_index(df)
A, edge_index = build_knn_graph(stations, k=3, weight="distance")
data = build_local_datasets(df, stations)
w0 = init_weights(data, seed=0)

# A2
h_a2 = run_a2(data, w0, A, alpha=1.0, n_iterations=100)

# A4 with K=1, 2, 3
h_a4_k1 = run_a4(data, w0, A, stations, alpha=1.0, K=1, n_iterations=500)
h_a4_k2 = run_a4(data, w0, A, stations, alpha=1.0, K=2, n_iterations=500)
h_a4_k3 = run_a4(data, w0, A, stations, alpha=1.0, K=3, n_iterations=500)

In [ ]:
import numpy as np

a2   = np.load("../results/a2.npz")
a4k1 = np.load("../results/a4_k1.npz")
a4k2 = np.load("../results/a4_k2.npz")
a4k3 = np.load("../results/a4_k3.npz")

# Example: plot val loss vs cumulative messages for all runs
a2["val_loss"]       # array of length 100
a2["cum_messages"]   # array of length 100
a4k1["val_loss"]     # array of length 1000

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load results
a2   = np.load("../results/a2.npz")
a4k1 = np.load("../results/a4_k1.npz")
a4k2 = np.load("../results/a4_k2.npz")
a4k3 = np.load("../results/a4_k3.npz")

runs = {
    "A2":      a2,
    "A4 K=1":  a4k1,
    "A4 K=2":  a4k2,
    "A4 K=3":  a4k3,
}

colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for (label, r), color in zip(runs.items(), colors):
    iters = np.arange(1, len(r["val_loss"]) + 1)
    axes[0].plot(iters, r["val_loss"], label=label, color=color)
    axes[1].plot(r["cum_messages"], r["val_loss"], label=label, color=color)

    # Horizontal test loss line
    axes[0].axhline(r["test_loss"], color=color, linestyle="--", linewidth=0.8, alpha=0.6)
    axes[1].axhline(r["test_loss"], color=color, linestyle="--", linewidth=0.8, alpha=0.6)

for ax, xlabel in zip(axes, ["Iteration", "Cumulative messages"]):
    ax.set_xlabel(xlabel)
    ax.set_ylabel("MSE")
    ax.legend()
    ax.grid(True, alpha=0.3)

# Shared legend note for dashed lines
fig.text(0.5, -0.02, "Dashed lines = final test MSE", ha="center", fontsize=9, color="gray")
axes[0].set_title("Convergence vs. iterations")
axes[1].set_title("Convergence vs. communication cost")

plt.tight_layout()
plt.savefig("../figures/convergence.pdf", bbox_inches="tight")
plt.show()